# About this notebook

This notebook contains my solution for **ARC Prize 2026 — ARC-AGI-3, Milestone 2**, built on [Tufa Labs’ Duck harness](https://www.kaggle.com/code/jeroencottaar/tufa-labs-duck-harness-june-30-milestone-winner). Full credit goes to Jeroen Cottaar and Tufa Labs for the original Duck solver, harness, and notebook.

The attached source bundle is an unchanged copy of Tufa Labs’ original bundle. **All my modifications to the harness are included in the patch at the beginning of this notebook**, which is applied automatically during setup. The patch is based on the original [Duck harness](https://github.com/Tufalabs/duck-harness) at commit `7652836`.

I also switched to [Qwen3.8-Flash-Next](https://huggingface.co/Qwen/Qwen3.8-Flash-Next), using [Intel’s AutoRound quantization](https://huggingface.co/Intel/Qwen3.8-Flash-Next-W4A16-AutoRound) and a separate quantized [MTP draft checkpoint](https://huggingface.co/albucino/Qwen3.8-Flash-Next-W4A16-FP8PLE) provided by Albucino for speculative decoding. Inference runs through [Pennyroyal, John Pezzulli’s SGLang fork](https://github.com/jpezzulli/sglang-rtxpro6000), v2.5.3, based on commit [`d00d88e`](https://github.com/jpezzulli/sglang-rtxpro6000/commit/d00d88efc8d6281b12be4f4073126aec95038c55), with contributions and patches from [Mamy Ratsimbazafy](https://github.com/mratsim/sglang-qwen38fn-sm120-turbo) and [Gabriel Olympie](https://github.com/gabrielolympie/sglang-flashnext-sm120), plus my own additional patches.

Third-party code and model weights remain subject to their respective licenses.

**Running your own copy:** After copying this notebook, manually select the **RTX PRO 6000** GPU before running it. Keep all attached source, wheelhouse, and model inputs enabled.

For more information, see my [GitHub repository](https://github.com/da-fr/arc-agi-3-solution).

## 0. Harness patch

Patch to add our modifications to the original tufa harness.

In [ ]:
# Harness patch (git diff bundle-base..HEAD), gzipped + base64 to fit Kaggle's 1 MB limit.
import base64, gzip, pathlib
pathlib.Path('/kaggle/harness-changes.patch').write_bytes(gzip.decompress(base64.b64decode(
"""
...
""")))
print('Writing /kaggle/harness-changes.patch')


## 1. Environment and submission mode

Detect whether this is a real competition rerun (which minimises diagnostics), set the
framework's environment flags, and put the CUDA libraries on the linker path.

In [ ]:
import json
import os
import pickle
import subprocess
import sys
import sysconfig
import time
from datetime import datetime, timedelta
from pathlib import Path
from urllib.request import urlopen

# Kaggle might cancel the run in no action is done in some timeframe.
# To prevent this, harness will start up even if backend is not yet ready,
# and the first started game will perform a single reset action.
NOTEBOOK_START_TIME = time.time()
SERVER_STARTUP_TIMEOUT = 12 * 60
os.environ['ARC3_WARMUP_ACTION_GAMES'] = '1'
os.environ['ARC3_HTTP_RETRY_INITIAL_SECONDS'] = '900'

# model and dataset paths
WHEELHOUSE_DIR    = '/kaggle/input/datasets/dfranzen/pennyroyal-v253'
MODEL_DIR         = '/kaggle/input/models/dfranzen/intel-qwen3.8-flash-next-w4a16-autoround/transformers/default/1'
DRAFT_MODEL_DIR   = '/kaggle/input/models/dfranzen/albucino-qwen3-8-flash-next-drafter/transformers/default/1'
ORIG_BUNDLE_DIR   = '/kaggle/input/datasets/dfranzen/taaf-kaggle-source-bundle-copy'
SERVED_MODEL_NAME = 'flashnext'

# Inputs mount as /kaggle/input/datasets/<owner>/<slug> (models: models/<owner>/<slug>/...)
# or, on some kernels, /kaggle/input/<slug>/...; use whichever exists, else say what is mounted.
INPUT_ROOT = Path('/kaggle/input')


def _resolve_input(path):
    p = Path(path)
    if p.exists():
        return str(p)
    rel = p.relative_to(INPUT_ROOT).parts  # (kind, [owner,] slug, *rest)
    # competitions/<slug>/... has no owner; datasets/ and models/ have one
    skip = 1 if rel and rel[0] == 'competitions' else 2
    candidates = [INPUT_ROOT.joinpath(*rel[skip:])] if len(rel) > skip else []
    for candidate in candidates:
        if candidate.exists():
            print(f'input {path} found at {candidate}')
            return str(candidate)
    mounted = sorted(str(q) for q in INPUT_ROOT.glob('*/*/*'))[:60]
    raise FileNotFoundError(f'input not mounted: {path} (also tried {candidates}); mounted: {mounted}')

WHEELHOUSE_DIR  = _resolve_input(WHEELHOUSE_DIR)
MODEL_DIR       = _resolve_input(MODEL_DIR)
DRAFT_MODEL_DIR = _resolve_input(DRAFT_MODEL_DIR)
ORIG_BUNDLE_DIR = _resolve_input(ORIG_BUNDLE_DIR)

# Apply harness patch
BUNDLE_DIR = Path('/kaggle/taaf-kaggle-source-share')
!rm -Rf $BUNDLE_DIR
!cp -a $ORIG_BUNDLE_DIR $BUNDLE_DIR
subprocess.run(["git", "apply", "--include=ARC3-Inference/*", "-v", "/kaggle/harness-changes.patch"], cwd=f"{BUNDLE_DIR}/src", check=True)
print('harness patch applied successfully')

# True only inside a real competition rerun; switches diagnostics + soft deadline.
TRUE_SUBMISSION = os.environ.get("KAGGLE_IS_COMPETITION_RERUN", "").strip().lower() in {"1", "true"}
NOTEBOOK_START_EPOCH = time.time()

# Non-interactive matplotlib backend: diagnostics render plots with no display attached.
os.environ["MPLBACKEND"] = "Agg"
# Marks the run as a (real or emulated) submission so the framework + solver can adjust.
os.environ["TAAF_RUN_AS_SUBMISSION"] = "1" if TRUE_SUBMISSION else "0"
# In submission, disable the periodic JSON/HTML diagnostics writes and per-frame logging.
os.environ["TAAF_MINIMAL_DIAGNOSTICS"] = "1" if TRUE_SUBMISSION else "0"
# Pin arc_agi's cached level_reset_only before its client is built (RESET keeps the level).
os.environ["ONLY_RESET_LEVELS"] = "true"

# Everything the run produces is written here.
WORKING_DIR = Path("/kaggle/working")
WORKING_DIR.mkdir(parents=True, exist_ok=True)
print(f"taaf.kaggle: TRUE_SUBMISSION={TRUE_SUBMISSION}")
SERVED_MODEL_HOST = '127.0.0.1'
SERVED_MODEL_PORT = 8001
SERVER_BASE_URL = f'http://{SERVED_MODEL_HOST}:{SERVED_MODEL_PORT}/v1'
USE_PRIORITY_SCHEDULING = True

setup_env = {
    ########## transformers config ##########
    'USE_TF': '0',
    'TRANSFORMERS_NO_TF': '1',
    'TRANSFORMERS_NO_TORCHVISION': '1',

    ########## serving config ##########
    'LOCAL_ANALYZER_BASE_URL': SERVER_BASE_URL,
    'OPENAI_BASE_URL': SERVER_BASE_URL,
    'LOCAL_ANALYZER_PROVIDER': 'vllm',
    'OPENAI_PROVIDER': 'vllm',
    'LOCAL_ANALYZER_MODEL_ID': SERVED_MODEL_NAME,
    'INFERENCE_ANALYZER_MODEL': SERVED_MODEL_NAME,
    'LOCAL_ANALYZER_APP_NAME': 'ARC3 Agent Harness',
    
    ########## analyzer config ##########
    'ARC3_REASONING_HISTORY_KEY': 'reasoning_content',
    'LOCAL_ANALYZER_ENABLE_THINKING': 'true',
    'LOCAL_ANALYZER_TOOL_STEPS': '0',
    'LOCAL_ANALYZER_TOOL_TIMEOUT': '30',
    'LOCAL_ANALYZER_YIELD_SECONDS': '0',
    'LOCAL_ANALYZER_YIELD_TOKENS': '2048',
    'LOCAL_ANALYZER_TEMPERATURE': '0.7',
    'LOCAL_ANALYZER_TOP_P': '0.95',
    'LOCAL_ANALYZER_TOP_K': '20',

    ########## context pruning ##########
    'LOCAL_ANALYZER_CONTEXT_WINDOW': (116+12)*1024,
    'LOCAL_ANALYZER_MAX_OUTPUT': 12*1024,
    'ARC3_CONTEXT_DRAIN_TOKENS': 58*1024,
    'ARC3_HISTORY_ASSISTANT_TURNS': 150,
    'ARC3_HISTORY_TURN_DRAIN': 30,
    'ARC3_HISTORY_DRAIN_COALESCE': '1',

    ########## tool output ##########
    'LOCAL_ANALYZER_TOOL_OUTPUT_TOKENS': '3072',
    'ARC3_MIDDLE_TRUNCATION': '1',
    'ARC3_ACTION_ECHO': '1',
    
    ########## prompt changes ##########
    'ARC3_EXPLAIN_GAMEPLAY_CHANGED': '1',
    'ARC3_NEW_CHANGED_PROMPTS': '1',
    'ARC3_LEVEL_TRANSFER_GUIDANCE': '1',
    'ARC3_ACTION_INFO': '1',
    'ARC3_DEDUPE_MULTICALL_LINE': '1',
    'ARC3_FRAME_DIFF_HINT': '1',

    ########## vision & animations ##########
    'MULTIMODAL_CONTEXT': 'current_grid',
    'MULTIMODAL_UPSCALE': '10',
    'ARC3_DIFF_IMAGE': '1',
    'ARC3_GAMEOVER_DIFF_IMAGE': '1',
    'ARC3_ANIMATION': '1',

    ########## guards ##########
    'ARC3_GUARDS_FROM_LEVEL': 2,
    'ARC3_BATCH_NOOP_BLOCK': '1',
    'ARC3_STALE_STATE_BLOCK': '1',

    ########## world model ##########
    'ARC3_MEMORY_SECTIONS': 'off',

    ########## persistent functions ##########
    'ARC3_PERSISTENT_FUNCTIONS': '1',
    'ARC3_PERSISTENT_FUNCTIONS_SCOPE': 'game',
    'ARC3_PERSISTENT_FUNCTIONS_IMPORTS': '1',

    ########## controls ##########
    'EXPOSE_UNDO': 'on',
}

if USE_PRIORITY_SCHEDULING: setup_env.update({
    'ARC3_MAX_ACTIVE_STREAMS': 10,
    'ARC3_PRIORITY_REFRESH_QUEUE': '1',
    'ARC3_PRIORITY_PACE': '0',
    'ARC3_PRIORITY_TAIL_FADE': '1',
    'ARC3_PRIORITY_TAIL_FADE_FRACTION': '0.2',
    'ARC3_PRIORITY_TAIL_LOOKUP': 'remaining',
    'ARC3_PRIORITY_SCORE_NORMALIZATION': '1',
})

# Arm overrides (build_notebook.py --env)
setup_env['ARC3_ANIM_DIGEST'] = '1'
setup_env['ARC3_ANIM_RING'] = '8'
setup_env['ARC3_ANIM_TRUTH'] = '1'
setup_env['ARC3_CONTEXT_DRAIN_TOKENS'] = '43008'
setup_env['ARC3_LEDGER_ON_TRIM'] = '1'
setup_env['ARC3_MAX_ACTIVE_STREAMS'] = '14'
setup_env['ARC3_OPENER_DEDUP'] = '1'
setup_env['EXPOSE_RESET'] = 'on'
setup_env['LOCAL_ANALYZER_CONTEXT_WINDOW'] = '98304'
setup_env['LOCAL_ANALYZER_TEMPERATURE'] = '0.65'
setup_env['TAAF_ARM'] = 'user-combo-10x'
os.environ.update({k: str(v) for k,v in setup_env.items()})

## 2. Precaching

Precache wheelhouse for faster startup.

In [ ]:
import os
import threading
import time

from concurrent.futures import as_completed, ThreadPoolExecutor

def precache(*paths, times=1, delay=0, threads=16) -> None:
    time.sleep(delay)
    print(f'Precaching: Started...')
    start_time = time.time()

    CHUNK_SIZE = 32 * 1024 * 1024  # 128 MB

    chunks = []  # list of (path, offset, length)
    total_size = 0
    file_count = 0

    def _add_file(i: int, path: str) -> None:
        nonlocal total_size, file_count
        size = os.path.getsize(path)
        total_size += size
        file_count += 1
        for offset in range(0, max(size, 1), CHUNK_SIZE):
            length = min(CHUNK_SIZE, size - offset)
            chunks.append((i, path, offset, length))

    for i, path in enumerate(paths):
        try:
            if os.path.isfile(path):
                _add_file(i, path)
                continue
            for root, _, files in os.walk(path):
                for file_name in files:
                    file_path = os.path.join(root, file_name)
                    if os.path.isfile(file_path):
                        _add_file(i, file_path)
        except: pass

    chunks.sort()

    _local = threading.local()

    def _read_chunk(chunk: tuple) -> None:
        try:
            buf = _local.buf
        except AttributeError:
            _local.buf = buf = bytearray(CHUNK_SIZE)
        i, path, offset, _ = chunk
        try:
            with open(path, 'rb') as f:
                f.seek(offset)
                remaining = CHUNK_SIZE
                for i in range(1024):
                    n = f.readinto(buf)
                    if not n:
                        break
                    remaining -= n
                    if remaining <= 0: break
        except: pass

    with ThreadPoolExecutor(max_workers=threads) as pool:
        for _ in range(times):
            list(pool.map(_read_chunk, chunks))

    elapsed = time.time() - start_time
    print(f'Precaching: loaded {file_count} files ({total_size / 1e9:.2f} GB) in {elapsed:.2f} seconds.\n')

precache_wheels_thread = threading.Thread(target=precache, args=(WHEELHOUSE_DIR,))
precache_wheels_thread.start()
precache_wheels_thread.join()

precache_model_thread = threading.Thread(target=precache, args=(MODEL_DIR, DRAFT_MODEL_DIR), kwargs={'delay': 60, 'threads': 3})

## 3. Install the ARC runtime

Install `arc-agi` from the offline competition wheelhouse (the Kaggle submission environment
has no internet).

In [ ]:
# Install the ARC runtime from the bundled competition wheels.
# Quiet: stdout is discarded; stderr (and a non-zero exit) still surface real failures.
subprocess.check_call(
    [
        sys.executable,
        "-m",
        "pip",
        "install",
        "--quiet",
        "--no-index",
        "--no-warn-conflicts",
        "--disable-pip-version-check",
        "--find-links",
        _resolve_input("/kaggle/input/competitions/arc-prize-2026-arc-agi-3/arc_agi_3_wheels"),
        "arc-agi",
    ],
    stdout=subprocess.DEVNULL,
)

## 4. Import the bundled source

Put the snapshotted repositories on the path (this process and any child processes).

In [ ]:
# Each bundled repo exposes its importable tree at <repo>/src or <repo>.
def _source_path_entries(bundle_dir: Path) -> list:
    entries = []
    for repo in sorted((bundle_dir / "src").iterdir(), reverse=True):
        for candidate in (repo / "src", repo):
            if candidate.is_dir():
                entries.append(candidate)
    return entries


# Environment handed to each setup command (paths + any keys it has persisted).
def _command_env() -> dict:
    env = os.environ.copy()
    # "$PYTHON" in a command resolves to this notebook's interpreter.
    env["PYTHON"] = sys.executable
    # Absolute path to the mounted source bundle.
    env["TAAF_KAGGLE_BUNDLE_DIR"] = str(BUNDLE_DIR)
    # The writable /kaggle/working directory.
    env["TAAF_KAGGLE_WORKING_DIR"] = str(WORKING_DIR)
    # A command writes a JSON object here to persist env keys to later commands + the run.
    return env


# Make the bundled repos importable here (sys.path) and in child processes (.pth).
source_entries = _source_path_entries(BUNDLE_DIR)
for entry in source_entries:
    sys.path.insert(0, str(entry))
pth_path = Path(sysconfig.get_paths()["purelib"]) / "taaf_kaggle_sources.pth"
pth_path.write_text("".join(f"{entry}\n" for entry in source_entries))
print(f"taaf.kaggle: wrote {pth_path} ({len(source_entries)} source roots)")

## 5. Start serving

Install and startup sglang to serve the model.

In [ ]:
# Paste this entire file into the launcher cell.
# Required notebook variables: MODEL_DIR, DRAFT_MODEL_DIR, WHEELHOUSE_DIR,
# WORKING_DIR, SERVED_MODEL_NAME, SERVED_MODEL_PORT.
# MODEL_DIR: Intel AutoRound INT4 target (BF16 PLE or albucino FP8 PLE).
# DRAFT_MODEL_DIR: downloaded albucino runtime/mtp-int4-g32, or its dataset root.
import os, sys, glob, time, shutil, subprocess, urllib.request, urllib.error
import base64, copy, gzip, hashlib, json, re, shlex, socket, zipfile
from pathlib import Path

PREFIX = "/tmp/sgl-intel"       # Separate from the previous fork's /tmp/sgl venv.
LOG = str(Path(WORKING_DIR) / "serve.log")
CFG = dict(
    CTX=(116+12+8)*1024,
    MEMFRAC=0.96,
    MAXREQ=10,
    CUDAGRAPH_MAXBS=10,
    # Extra decode batch sizes to capture CUDA graphs for, on top of the fixed
    # 1/2/4/7/8/9/10 below. Without a graph, batch 3 pads to 4 and 5/6 pad to 7.
    # [] keeps the launch arguments exactly as Franzen shipped them.
    CUDAGRAPH_BS_EXTRA=[],
    MAMBA_CACHE=60,
    CHUNK=8192,
    MAX_PREFILL=16384,
    SPEC=True,
    FRSPEC=True,
    AUTOTUNE=True,
    # Preserve the original Kaggle checkpoint prefetch/cache behavior.
    PREFETCH_CHECKPOINTS=True,
    DROP_CACHE_AFTER_LOAD=True,
    # JIT caches (issue #6; serving/CUSTOM.md). JIT_CACHE_SAVE: once the server is
    # ready, write them to WORKING_DIR/sgl-jit-cache.tar plus a manifest.
    # JIT_CACHE_DIR: a mounted dataset holding that tar; it is restored before
    # launch only if its manifest matches this runtime and config.
    JIT_CACHE_SAVE=False,
    JIT_CACHE_DIR=None,
    LINEAR_BACKEND="flashinfer",
    KVDTYPE="fp8_e4m3",
    SSM_DTYPE="bfloat16",
    MAMBA_RADIX="extra_buffer",
    GDN_MTP_CACHE_MODE="none",
    SERVED_NAME=SERVED_MODEL_NAME,
    SPEC_ACCEPT_SINGLE=1.0,
    SPEC_ACCEPT_ACC=1.0,
    SPEC_STEPS=3,
)
CFG.setdefault("DRAFT_KVDTYPE", CFG["KVDTYPE"])  # before arm overrides
CFG.setdefault("ATTN_BACKEND", "")  # "" keeps SGLang's default (flashinfer)
CFG.setdefault("TEXT_CONFIG_OVERRIDE", {})  # {} serves MODEL_DIR unchanged
CFG.setdefault("SPEC_ADAPTIVE", {})  # {} keeps a fixed SPEC_STEPS
# Arm overrides (build_notebook.py --serve); only existing keys.
_SERVE_OVERRIDES = {
    'CTX': 106496,
    'CUDAGRAPH_BS_EXTRA': [3, 5, 6, 11, 12, 13],
    'CUDAGRAPH_MAXBS': 14,
    'MAMBA_CACHE': 84,
    'MAXREQ': 14,
    'MEMFRAC': 0.96,
}
assert set(_SERVE_OVERRIDES) <= set(CFG), set(_SERVE_OVERRIDES) - set(CFG)
CFG.update(_SERVE_OVERRIDES)
VENV = f"{PREFIX}/venv"
PYTHON = f"{VENV}/bin/python"
SGLANG = f"{VENV}/bin/sglang"
TOKEN_MAP_SHA = "becfa41d394b86c26c632bea8f3c6ea64bbb76d7b238d8673c06afae21269f25"
TOKENIZER_SHA = "06b9509352d2af50381ab2247e083b80d32d5c0aba91c272ca9ff729b6a0e523"
EXPERT_TARGET = r"re:^mtp\.layers\.0\.mlp\.experts\.[0-9]+\.(gate_proj|up_proj|down_proj)$"
DENSE_IGNORE = r"re:^(?!mtp\.layers\.0\.mlp\.experts(?:\.|$)).*"


def run(cmd, *, check=True, env=None):
    cmd = list(map(str, cmd))
    print("$", shlex.join(cmd))
    result = subprocess.run(cmd, text=True, capture_output=True, env=env)
    if result.stdout.strip(): print(result.stdout.strip()[-6000:])
    if result.stderr.strip() and result.returncode: print(result.stderr.strip()[-6000:])
    if check and result.returncode:
        raise RuntimeError(f"Command failed ({result.returncode}): {shlex.join(cmd)}")
    return result


def find_unique(root, name, *, directory=False, required=True):
    root = Path(root)
    direct = root if root.name == name else root / name
    valid = lambda p: p.is_dir() if directory else p.is_file()
    if valid(direct): return direct
    hits = sorted(p for p in root.rglob(name) if valid(p))
    if len(hits) > 1:
        raise RuntimeError(f"Multiple {name} matches under {root}; use a more specific directory: {hits}")
    if hits: return hits[0]
    if required: raise FileNotFoundError(f"{name} not found under {root}")
    return None


def sha256(path):
    h = hashlib.sha256()
    with Path(path).open("rb") as f:
        for block in iter(lambda: f.read(8*1024*1024), b""): h.update(block)
    return h.hexdigest()


def indexed_shards(root):
    root = Path(root)
    index = json.loads((root / "model.safetensors.index.json").read_text())
    names = sorted(set(index["weight_map"].values()))
    if not names: raise RuntimeError(f"Empty weight index: {root}")
    for name in names:
        rel = Path(name)
        if rel.is_absolute() or ".." in rel.parts:
            raise RuntimeError(f"Invalid shard path: {name}")
        p = root / rel
        if not p.is_file() or p.stat().st_size < 8:
            raise RuntimeError(f"Missing/empty shard: {p}")
    return index, names


def prepare_draft_view(target, source_root, work_root):
    """Adapt metadata only; link original dataset weights without copying them."""
    target, source_root, work_root = map(Path, (target, source_root, work_root))
    configs = [source_root / "config.json"] if (source_root / "config.json").is_file() else []
    configs += sorted(p for p in source_root.rglob("config.json") if p not in configs)
    candidates = []
    for path in configs:
        cfg = json.loads(path.read_text())
        q = cfg.get("quantization_config", {})
        if q.get("quant_method") == "compressed-tensors" and "mtp_routed_experts" in q.get("config_groups", {}):
            candidates.append((path.parent, cfg))
    if len(candidates) != 1:
        raise RuntimeError(f"Expected one INT4 g32 MTP checkpoint under DRAFT_MODEL_DIR={source_root}; found {len(candidates)}")
    source, cfg = candidates[0]
    index, shards = indexed_shards(source)
    if not any(k.startswith("mtp.layers.0.mlp.experts.") for k in index["weight_map"]):
        raise RuntimeError("Draft index does not contain the expected MTP experts")
    cfg = copy.deepcopy(cfg)
    q = cfg["quantization_config"]
    group = q["config_groups"]["mtp_routed_experts"]
    w = group["weights"]
    if (w["num_bits"], w["group_size"], w["symmetric"]) != (4, 32, True):
        raise RuntimeError("Expected symmetric INT4 group32 draft experts")
    if group["targets"] not in (["RoutedExperts"], [EXPERT_TARGET]):
        raise RuntimeError(f"Unexpected draft target rules: {group['targets']}")
    if q.get("ignore", []) not in ([], [DENSE_IGNORE]):
        raise RuntimeError("Unexpected draft ignore rules")
    group["targets"] = [EXPERT_TARGET]
    q["ignore"] = [DENSE_IGNORE]
    # Content/path-specific directory makes notebook reruns idempotent without
    # overwriting an unrelated view. We never modify the read-only datasets.
    identity = json.dumps({"source": str(source.resolve()), "target": str(target.resolve()), "config": cfg}, sort_keys=True)
    suffix = hashlib.sha256(identity.encode()).hexdigest()[:16]
    view = work_root / f"draft-view-{suffix}"
    view.mkdir(parents=True, exist_ok=True)
    config_path = view / "config.json"
    config_text = json.dumps(cfg, indent=2) + "\n"
    if config_path.exists() or config_path.is_symlink():
        if config_path.is_symlink() or config_path.read_text() != config_text:
            raise RuntimeError(f"Unexpected existing draft-view config: {config_path}")
    else:
        config_path.write_text(config_text)
    links = {name: source / name for name in shards}
    links["model.safetensors.index.json"] = source / "model.safetensors.index.json"
    for name in ("tokenizer.json", "tokenizer_config.json", "chat_template.jinja", "preprocessor_config.json", "generation_config.json"):
        if (target / name).is_file(): links[name] = target / name
    for name, src in links.items():
        dest = view / name
        dest.parent.mkdir(parents=True, exist_ok=True)
        if dest.exists() or dest.is_symlink():
            if not dest.is_symlink() or dest.resolve() != src.resolve():
                raise RuntimeError(f"Unexpected existing draft-view file: {dest}")
        else:
            dest.symlink_to(src.resolve())
    print(f"Draft source: {source}\nSGLang draft view: {view} ({len(shards)} linked weight shards)")
    return view


# JIT cache save/restore helpers (issue #6), used only when CFG JIT_CACHE_SAVE or
# JIT_CACHE_DIR is set: serving/kaggle/jit_cache.py, loaded as the module
# `jit_cache`. kaggle/build_notebook.py embeds it in place of this placeholder.
import types
jit_cache = types.ModuleType("jit_cache")
exec(compile(r'''"""Save and restore SGLang's JIT caches across Kaggle containers (issue #6).

Every Kaggle container starts with empty Triton / FlashInfer / SGLang-JIT /
TorchInductor / CUDA / TileLang caches, so the first CUDA-graph capture compiles
from scratch (~145 s). The launcher (serving/kaggle/launch_sglang.py) uses this
module to snapshot those caches into one tar once the server is ready
(CFG JIT_CACHE_SAVE) and to restore a snapshot from a mounted dataset before
launch (CFG JIT_CACHE_DIR). A snapshot is restored only when its manifest
matches this runtime (GPU, CUDA, torch, SGLang and kernel package versions)
and this serving config; otherwise the launch stays cold.

kaggle/build_notebook.py embeds this file into the launcher cell as the module
``jit_cache``, so it must stay plain Python with no imports from the repo (and
no ``from __future__`` line, which is only legal at the top of a file).
Nothing here needs a GPU: the runtime probe is a code string the launcher runs
in the SGLang venv.
"""
import datetime
import hashlib
import io
import json
import os
import shutil
import tarfile
import time
from pathlib import Path, PurePosixPath

KIND = "sgl-jit-cache"
FORMAT = 1
TAR_NAME = "sgl-jit-cache.tar"
# Written next to the tar and also stored inside it, so a tar that Kaggle
# unpacked on upload still carries its manifest at the root of the tree.
MANIFEST_NAME = "sgl-jit-cache.manifest.json"
# Directories under the launcher's cache root (PREFIX/cache) that hold compiled kernels.
CACHE_SUBDIRS = ("triton", "flashinfer", "sglang/jit", "torchinductor", "cuda")
TILELANG_KEY = "tilelang"  # ~/.tilelang
# Runtime fields recorded for information only; every other field must match.
INFO_ONLY = ("driver",)
# CFG keys that cannot change a compiled kernel; they are left out of the
# config hash so a cache saved by one startup variant serves the others. The
# MTP acceptance thresholds are plain float arguments of sgl-kernel's prebuilt
# tree_speculative_sampling_target_only, called outside the CUDA graphs (SGLang
# even lets set_internal_state change them on a running server).
CFG_IGNORED = ("JIT_CACHE_SAVE", "JIT_CACHE_DIR", "PREFETCH_CHECKPOINTS", "DROP_CACHE_AFTER_LOAD",
               "SPEC_ACCEPT_SINGLE", "SPEC_ACCEPT_ACC")
# Server environment variables that select or configure kernels.
ENV_PREFIXES = ("SGLANG_", "FLASHINFER_", "TRITON_", "TORCHINDUCTOR_", "TORCH_CUDA_ARCH_LIST")
# Run with the SGLang venv's python; prints one line, PROBE_MARKER + JSON.
PROBE_MARKER = "JIT_CACHE_RUNTIME "
RUNTIME_PROBE = r"""
import importlib.metadata, json, os, platform, subprocess
import torch
info = {
    "python": platform.python_version(),
    "torch_cuda": torch.version.cuda,
    "gpu": torch.cuda.get_device_name(0),
    "capability": ".".join(map(str, torch.cuda.get_device_capability(0))),
}
for dist in ("torch", "sglang", "sgl-kernel", "triton", "flashinfer-python", "flashinfer-cubin",
             "flashinfer-jit-cache", "tilelang", "apache-tvm-ffi"):
    try:
        info[dist] = importlib.metadata.version(dist)
    except importlib.metadata.PackageNotFoundError:
        info[dist] = None
def _out(cmd):
    try:
        return subprocess.run(cmd, capture_output=True, text=True, timeout=60).stdout
    except Exception:
        return ""
releases = [l.strip() for l in _out([os.environ.get("CUDACXX", "nvcc"), "--version"]).splitlines() if "release" in l]
info["nvcc"] = releases[-1] if releases else None
driver = _out(["nvidia-smi", "--query-gpu=driver_version", "--format=csv,noheader"]).split()
info["driver"] = driver[0] if driver else None
print("JIT_CACHE_RUNTIME " + json.dumps(info, sort_keys=True))
"""


def parse_probe(stdout):
    """The runtime dict printed by RUNTIME_PROBE."""
    for line in reversed(stdout.splitlines()):
        if line.startswith(PROBE_MARKER):
            return json.loads(line[len(PROBE_MARKER):])
    raise ValueError("runtime probe printed no JIT_CACHE_RUNTIME line")


def config_snapshot(cfg, env, model_config=None):
    """The launch settings that can change compiled kernels, as plain JSON data.

    ``model_config``: the target checkpoint's config.json bytes, hashed so a
    different model does not reuse the cache.
    """
    snapshot = {
        "cfg": {k: cfg[k] for k in sorted(cfg) if k not in CFG_IGNORED},
        "env": {k: env[k] for k in sorted(env) if k.startswith(ENV_PREFIXES)},
    }
    if model_config is not None:
        snapshot["model"] = {"config.json sha256": hashlib.sha256(model_config).hexdigest()}
    return snapshot


def config_hash(snapshot):
    blob = json.dumps(snapshot, sort_keys=True, default=str).encode()
    return hashlib.sha256(blob).hexdigest()[:16]


def cache_roots(cache_dir, home):
    """{key inside the tar: directory on this machine} for every cached directory."""
    roots = {f"cache/{name}": Path(cache_dir, name) for name in CACHE_SUBDIRS}
    roots[TILELANG_KEY] = Path(home, ".tilelang")
    return roots


def make_manifest(runtime, snapshot, roots, label=None):
    return {
        "kind": KIND,
        "format": FORMAT,
        "runtime": dict(runtime),
        "config_hash": config_hash(snapshot),
        "config": snapshot,
        "roots": {key: str(path) for key, path in sorted(roots.items())},
        "label": label,
        "created": datetime.datetime.now(datetime.timezone.utc).isoformat(timespec="seconds"),
    }


def mismatches(saved, current):
    """Why a saved manifest cannot be used here; empty when it matches."""
    problems = []
    for key in ("kind", "format"):
        if saved.get(key) != current.get(key):
            problems.append(f"{key}: saved {saved.get(key)!r}, here {current.get(key)!r}")
    if problems:
        return problems
    saved_rt, current_rt = saved.get("runtime", {}), current.get("runtime", {})
    for key in sorted(set(saved_rt) | set(current_rt)):
        if key not in INFO_ONLY and saved_rt.get(key) != current_rt.get(key):
            problems.append(f"{key}: saved {saved_rt.get(key)!r}, here {current_rt.get(key)!r}")
    if saved.get("config_hash") != current.get("config_hash"):
        differ = []
        saved_config, current_config = saved.get("config", {}), current.get("config", {})
        for section in sorted(set(saved_config) | set(current_config)):
            a, b = saved_config.get(section, {}), current_config.get(section, {})
            differ += [f"{section}.{k}" for k in sorted(set(a) | set(b)) if a.get(k) != b.get(k)]
        problems.append(
            f"config_hash: saved {saved.get('config_hash')}, here {current.get('config_hash')}"
            + (f" ({', '.join(differ)})" if differ else "")
        )
    return problems


def _skipped(name):
    # Lock files would block the next build (torch's FileBaton waits while one
    # exists); tmp.pid_* are Triton's half-written entries and .staging-* are
    # SGLang JIT builds still in progress.
    return (name == "lock" or name.endswith(".lock") or name.startswith("tmp.pid_")
            or name.startswith(".staging-"))


def iter_cache_files(roots):
    """Yield (arcname, path) for every regular cache file, in a stable order."""
    for key, root in sorted(roots.items()):
        root = Path(root)
        if not root.is_dir() or root.is_symlink():
            continue
        for dirpath, dirnames, filenames in os.walk(root):
            dirnames[:] = sorted(d for d in dirnames if not _skipped(d))
            for name in sorted(filenames):
                path = Path(dirpath, name)
                if _skipped(name) or path.is_symlink() or not path.is_file():
                    continue
                yield f"{key}/{path.relative_to(root).as_posix()}", path


def _write_atomic(path, data):
    tmp = path.with_name(path.name + ".partial")
    try:
        tmp.write_bytes(data)
        os.replace(tmp, path)
    except BaseException:
        tmp.unlink(missing_ok=True)
        raise


def _manifest_bytes(manifest):
    return (json.dumps(manifest, indent=2, sort_keys=True) + "\n").encode()


def _size(path):
    """The file's size, or 0 if it has gone since it was listed."""
    try:
        return path.stat().st_size
    except FileNotFoundError:
        return 0


def save(tar_path, roots, manifest, reserve=1 << 30):
    """Write the cache files under ``roots`` to ``tar_path`` plus a manifest beside it.

    Returns the manifest as written (with file and byte counts). The tar holds
    each root's files under its key, with their modification times preserved,
    and then the manifest at its root. A file removed after the listing (a
    build finishing) is left out. Refuses (OSError) when the disk would keep
    less than ``reserve`` bytes free. An error while writing the tar removes
    its partial file, so nothing half-written is left in the output.
    """
    tar_path = Path(tar_path)
    files = list(iter_cache_files(roots))
    content = sum(_size(p) for _, p in files)
    free = shutil.disk_usage(tar_path.parent).free
    if content + reserve > free:
        raise OSError(f"{content} bytes of cache would leave less than {reserve} of {free} free bytes")
    counts = {key: 0 for key in roots}
    added, content = 0, 0
    tmp = tar_path.with_name(tar_path.name + ".partial")
    try:
        with tarfile.open(tmp, "w", format=tarfile.PAX_FORMAT) as tar:
            for arcname, path in files:
                try:
                    size = path.stat().st_size
                    # Raises before writing anything when the file has gone.
                    tar.add(str(path), arcname=arcname, recursive=False)
                except FileNotFoundError:
                    continue
                counts[next(k for k in roots if arcname.startswith(k + "/"))] += 1
                added, content = added + 1, content + size
            # Last, so its counts are of the files actually stored.
            manifest = dict(manifest, files=added, content_bytes=content, files_per_root=counts)
            inner = _manifest_bytes(manifest)
            info = tarfile.TarInfo(MANIFEST_NAME)
            info.size, info.mtime, info.mode = len(inner), int(time.time()), 0o644
            tar.addfile(info, io.BytesIO(inner))
        manifest["tar_bytes"] = tmp.stat().st_size
        os.replace(tmp, tar_path)
    except BaseException:
        tmp.unlink(missing_ok=True)
        raise
    _write_atomic(tar_path.with_name(MANIFEST_NAME), _manifest_bytes(manifest))
    return manifest


def candidate_dirs(path, input_root="/kaggle/input"):
    """``path`` plus its other Kaggle mount layout.

    Datasets mount as /kaggle/input/datasets/<owner>/<slug> on some kernels and
    as /kaggle/input/<slug> on others (the same fallback the notebook's
    _resolve_input uses).
    """
    path, input_root = PurePosixPath(path), PurePosixPath(input_root)
    candidates = [path]
    try:
        rel = path.relative_to(input_root).parts
    except ValueError:
        return candidates
    if len(rel) >= 3 and rel[0] == "datasets":
        candidates.append(input_root.joinpath(*rel[2:]))
    return candidates


def _read_json(path):
    return json.loads(Path(path).read_text())


def _tar_manifest(tar_path):
    with tarfile.open(tar_path) as tar:
        member = tar.getmember(MANIFEST_NAME)
        return json.loads(tar.extractfile(member).read())


def locate(path):
    """Find a saved cache under a mounted directory.

    Returns None, or {"kind": "tar" | "tree", "manifest", "path"}. "tar" is the
    archive as uploaded; "tree" is the root of an archive Kaggle unpacked on
    upload (it holds the manifest and the cached directories).
    """
    for base in map(Path, candidate_dirs(path)):
        if not base.is_dir():
            continue
        tars = sorted(p for p in base.rglob(TAR_NAME) if p.is_file())
        if len(tars) > 1:
            raise ValueError(f"several {TAR_NAME} under {base}: {tars}")
        if tars:
            sidecar = tars[0].with_name(MANIFEST_NAME)
            manifest = _read_json(sidecar) if sidecar.is_file() else _tar_manifest(tars[0])
            return {"kind": "tar", "manifest": manifest, "path": tars[0]}
        trees = []
        for manifest_path in sorted(base.rglob(MANIFEST_NAME)):
            if not manifest_path.is_file():
                continue
            manifest = _read_json(manifest_path)
            keys = manifest.get("roots", {})
            if any((manifest_path.parent / key).is_dir() for key in keys):
                trees.append({"kind": "tree", "manifest": manifest, "path": manifest_path.parent})
        if len(trees) > 1:
            raise ValueError(f"several unpacked caches under {base}: {[t['path'] for t in trees]}")
        if trees:
            return trees[0]
    return None


def _check_members(members, keys):
    """Refuse anything but plain files and directories under a known root."""
    for m in members:
        parts = PurePosixPath(m.name).parts
        if m.name.startswith("/") or ".." in parts or not (m.isfile() or m.isdir()):
            raise ValueError(f"unsafe tar member: {m.name!r}")
        if m.name != MANIFEST_NAME and not any(m.name == k or m.name.startswith(k + "/") for k in keys):
            raise ValueError(f"tar member outside the cache roots: {m.name!r}")


def _stage_tar(tar_path, staging, keys):
    with tarfile.open(tar_path) as tar:
        members = tar.getmembers()
        _check_members(members, keys)
        kwargs = {"filter": "data"} if hasattr(tarfile, "data_filter") else {}
        tar.extractall(staging, members=[m for m in members if m.name != MANIFEST_NAME], **kwargs)


def _stage_tree(tree, staging, keys):
    for key in keys:
        src = Path(tree, key)
        if src.is_dir():
            shutil.copytree(src, Path(staging, key), symlinks=False, dirs_exist_ok=True)


def _install(staged, dest):
    """Move a staged root into place, merging into an existing directory."""
    dest.parent.mkdir(parents=True, exist_ok=True)
    if dest.is_dir() and not any(dest.iterdir()):
        dest.rmdir()
    if not dest.exists():
        shutil.move(str(staged), str(dest))
        return
    for dirpath, _, filenames in os.walk(staged):
        target_dir = dest / Path(dirpath).relative_to(staged)
        target_dir.mkdir(parents=True, exist_ok=True)
        for name in filenames:
            shutil.move(str(Path(dirpath, name)), str(target_dir / name))


def restore(source, roots, current, staging):
    """Restore a matching cache from ``source`` into ``roots``.

    Returns (restored, message). Nothing is installed unless the manifest
    matches ``current`` and the whole archive unpacked cleanly into
    ``staging``, so a mismatched, truncated or unsafe cache leaves the launch
    cold. Installing then moves each staged root into place.
    """
    found = locate(source)
    if found is None:
        tried = ", ".join(map(str, candidate_dirs(source)))
        return False, f"no {TAR_NAME} or unpacked cache under {tried}"
    saved = found["manifest"]
    problems = mismatches(saved, current)
    if problems:
        return False, f"{found['path']} does not match this runtime: " + "; ".join(problems)
    if found["kind"] == "tar" and saved.get("tar_bytes") not in (None, found["path"].stat().st_size):
        return False, f"{found['path']} is {found['path'].stat().st_size} bytes, manifest says {saved['tar_bytes']}"
    staging = Path(staging)
    shutil.rmtree(staging, ignore_errors=True)
    staging.mkdir(parents=True)
    keys = sorted(roots)
    try:
        (_stage_tar if found["kind"] == "tar" else _stage_tree)(found["path"], staging, keys)
        files = sum(len(f) for key in keys for _, _, f in os.walk(staging / key))
        for key in keys:
            if (staging / key).is_dir():
                _install(staging / key, Path(roots[key]))
    finally:
        shutil.rmtree(staging, ignore_errors=True)
    return True, f"restored {files} files from {found['path']} ({found['kind']}, saved {saved.get('created')})"


def unsaved_dirs(cache_dir, roots):
    """[(directory, bytes)] for files under ``cache_dir`` that a snapshot leaves out.

    The save run prints this, so kernels the server caches somewhere else show up.
    """
    saved = [Path(p) for p in roots.values()]
    report = []
    for child in sorted(Path(cache_dir).iterdir()) if Path(cache_dir).is_dir() else []:
        if not child.is_dir() or child in saved:
            continue
        size = sum(p.stat().st_size for p in child.rglob("*")
                   if p.is_file() and not p.is_symlink() and not any(p.is_relative_to(s) for s in saved))
        if size:
            report.append((str(child), size))
    return report
''', "serving/kaggle/jit_cache.py", "exec"), jit_cache.__dict__)


# ---- validate notebook inputs / identify the offline bundle ----
MODEL_DIR = str(Path(MODEL_DIR).expanduser().resolve())
assert Path(MODEL_DIR).is_dir(), f"model dir not found: {MODEL_DIR}"
assert CFG["CUDAGRAPH_MAXBS"] >= CFG["MAXREQ"], "CUDAGRAPH_MAXBS must cover MAXREQ"
assert all(isinstance(n, int) and 1 <= n <= CFG["CUDAGRAPH_MAXBS"] for n in CFG["CUDAGRAPH_BS_EXTRA"]), \
    "CUDAGRAPH_BS_EXTRA: batch sizes from 1 to CUDAGRAPH_MAXBS"
assert CFG["CHUNK"] > 0 and CFG["CHUNK"] % 64 == 0
if CFG["JIT_CACHE_SAVE"] or CFG["JIT_CACHE_DIR"]:
    assert jit_cache is not None, "CFG JIT_CACHE_* need the jit_cache module that kaggle/build_notebook.py embeds"
# Refuse to start a second server on an occupied port; do not kill other jobs.
with socket.socket() as probe:
    if probe.connect_ex(("127.0.0.1", int(SERVED_MODEL_PORT))) == 0:
        raise RuntimeError(f"Port {SERVED_MODEL_PORT} is occupied; stop the existing server before rerunning")
Path(PREFIX).mkdir(parents=True, exist_ok=True)
Path(LOG).parent.mkdir(parents=True, exist_ok=True)
WHEELS = find_unique(WHEELHOUSE_DIR, "wheels", directory=True)
LOCK = find_unique(WHEELHOUSE_DIR, "requirements.lock")
sglang_wheels = sorted(WHEELS.glob("sglang-*.whl"))
assert len(sglang_wheels) == 1, f"Expected one SGLang wheel, found {sglang_wheels}"
wheel = sglang_wheels[0]
install_marker = Path(PREFIX) / "installed-bundle.txt"

# ---- one-time offline install, isolated from the old mratsim environment ----
install_env = dict(os.environ, UV_OFFLINE="1", UV_PYTHON_DOWNLOADS="never", PYTHONNOUSERSITE="1")
install_env.pop("PYTHONPATH", None)
if not Path(SGLANG).exists() or not install_marker.is_file():
    print("=== installing Pennyroyal (offline) ===")
    # Bootstrap uv directly from its wheel; no global pip changes/ensurepip.
    uv_wheels = sorted(WHEELS.glob("uv-*.whl"))
    assert len(uv_wheels) == 1, "Expected one bundled uv wheel"
    uv_binary = Path(PREFIX) / "bin/uv"
    uv_binary.parent.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(uv_wheels[0]) as archive:
        members = [n for n in archive.namelist() if Path(n).name == "uv" and not n.endswith("/")]
        assert len(members) == 1, "Cannot identify the Linux uv executable in its wheel"
        binary = archive.read(members[0])
        assert binary.startswith(b"\x7fELF"), "Bundled uv executable is not a Linux ELF binary"
        uv_binary.write_bytes(binary)
    uv_binary.chmod(0o755)
    uv = [str(uv_binary)]
    if not Path(PYTHON).exists():
        run(uv + ["venv", "--python", sys.executable, VENV], env=install_env)
    install = uv + ["pip", "install", "--python", PYTHON, "--no-index", "--find-links", str(WHEELS)]
    run(install + ["-r", str(LOCK)], env=install_env)
    run(install + ["--reinstall", "--no-deps", str(wheel)], env=install_env)
    install_marker.write_text('install')
else:
    print(f"Matching offline bundle already installed at {VENV}")

# ---- our runtime patches to the installed SGLang (serving/runtime_patches/) ----
# kaggle/build_notebook.py embeds every file of serving/runtime_patches/ here:
# *.patch is a unified diff against site-packages (paths like sglang/srt/...),
# applied with `patch -p1`; *.py is run as `python <file> <site-packages>`.
# Each is applied once per install (a marker records its hash). Empty = no-op.
SGLANG_RUNTIME_PATCHES = json.loads(gzip.decompress(base64.b64decode(
"""
H4sIAAAAAAACE61YbW/buhn9K5zvh8h3thJ32Idl9QVykzQL2iRFmt4NiAuGkWhbsEQKJBXXC/Lfdx6SsmXH7VrgJkAiUeLh88bzHOq59+ZoWGk5NLpxhZql
9ap3zHoT/3vbKFdUktXCZfNjdqXPWXyNPQlTCOUsm2rD3FyyQlknylLm7NPFB4E33qR/T/+WTtREAYY9rvxbpWhUNpeGCcse6pWba8VeGcDe2sLJYS2yhZhJ
+9tDyi4dE3UtVW6ZmKiloRuAYI7K2YOVpcwcl18x6OwDbGF2VsKIQ2vcYSlW0thDrHHodL3AArDq4U7X71MYvxQm51mTiwdWar2wMFM4pgTcbmqGywxeMQrD
gFntnWiXt1JaJp+kWU3IeZUDimGJ4WIdplxmhS20GsCmrGxyGkOkilyy089nJ8MZoOZYonaNkT5Yd50Fco0FlHZzmtaoUlpL0T25vuAnt6f86uac3958vru8
vmCFhTmOPCcLrTQw68BOlFRPhdGqksqxRCiErDqwbNLzb3A8nfT6KcOiihWOGZnpqm6ctMFRWczmyLGe0i3AQoD9M/KQUlAaKfIVy+baIkJTo6vOYyyll0iG
nrpKfGW10Y/isSgLV0h7TN4y9iis5EqbirElL9iY1fh7yBLbVHFdCumu27+ffDrn77cB+ynhdX7+Xbg5q7SRrDWcgur0QlKURAhVpXNZsiXq0RlRKNTvEtN2
kBLVVG15cfzlAGFPhWB35/+546c31+8uL/jNH+e3t5dn5/2BB9ammBVKlDtQrSkLKWvciMyVfmsUZmNADDsqyiPJr3jCtEJWRJ7v4IWptqIqNWLZCckKUyh2
A783kSWKZ9icyEsup6Ip3Q6akZQKURZWOJQt008+xyVbDNhyXmRzZuemUItY9zsm+wSoJitlYwnN+0geTOVSWrf2fUnFsmUpigi0YHes2Un6x5Pbk6u2KHzB
KVYJ7IkE27SUwrrvz7+6vObvkZ3/ShM2siGjKMobt6X3wcmqjhhUlLC01oYCIkAF2pfor7+yZIRC3Wcj1ugg5ut9HfxnBbEYqpKoEMmQuWeWuPNoJ/stTAwk
wnWm1bSYgSFytgDYbSQXvFnKqQOO0w3CBxyqXeZrZeiZu9QzViJDlD1pKGchH7CtKYUnmIkCwxC7lqtjkI0atrvVZihhNRuwGTZzLSO1DQCPB2AKXyCPhbB+
qLFOVxPVEt8UzjlPfNMGIUDZCDI/VsAg8AOGwa5+DXQREVaPRDpRgUnxHsLiLRYuZWdGwOPk6u5j3/ejx1JnixBOuyhqMjMQ5UTtZObs9uTd3XhEyQjtTXmy
4nzaEPlyzoqKsoyCQEB8+VtKXBy1KxtnoBvOy+Kxff2jILqYqKuT2/egr0nvFzbED8O6+ztmQtyLkUMT2iv37VXaw1eNsO+RyNST69N/3dx+An4y6WHrsu2W
h8EBVs5K2g2fYgSpw9346CXXKKb8rqlL2T/uvkqvnPriwnCfvPj9w80puXFwcDBRf7onbTC1pdLmwmRc+yDHLJ2dY+k4nMbWlc6kg3/7+15whnqYdaaok35a
6qU0ST9Cxj4xBgW65IeAw4yAOzoCcgvl2eNnkPyECNTBCSw2ZtNSix9E8lMC0lH6jw6Wr+kfDZl/OaLsiRhph8SbSm8408hwtQJF+AutfI14vA83FxfnZ1gZ
oiPxhTNRVJbeEL+3eexiCWmu9mbA/B0IMDKA4WCogh5QHx4ENQCOqQpF/2phRIWS9bQetu21XK4bZKs9276yJUYUOfTQrveAYiVGBQ/VEkohUgDhxpp02mTz
oEkYFs5zUMl4bS97y44Y+6UVt5UwYJ34llcU9sDzuifg4SgiY2ADgUCrGeKNrVfVCTwcH/XjauiE4U2YkEb+TbYClIZyQWfJi2o8HLUz17NmoCGk0UMN2GhA
i8eXFqSqUu/5/XD0Jdo29SFnY6KstQib9I43PZRG1/jkRVL7zCRtlhYbc9InUTbSppBtSRga+PZP13eopf4GdknmoHMSSggGJ9CRHL5pAyLLLfOipNgyztbU
NHI0tE3cjCML2+VzaTOcGEBAuxY48HtJxVt/295vWObjIqckKsHFAMiaaoMBmgNoH86FJVAwXqBEzVIKM4PBsVw3gLQwJ/N9yUX4t6H69751n6bpgB0zKpNR
yEncMYt+/wswyIXtmeswkfqxvCwWMukggg4y4VBvhOfjOuguuJO+gOQlhQfBTh28hq/7O9OWCMzyp0O+UwykzbYqwdfTq7lprZfJKD3CkoFFfsqUdmUrOwtB
62JD/EGFfm6MNsl00vusFoqOOPvb0/iZ7P6LefHMuS96gUIiL0Ze2+z1ZZxlJCSKgsFOb/FpmrtVLXf5l7RckYnHUiagkAHLprOWQyNQsnEKx1N/eM3iy6/V
QydypJXRW1AoJgEqdYWoEjkJvtAnIn/B5ZCw9f02DqnOHSwoRR7FJic3Ce+dQBK+b8JGjnKSozTrGgq4TwKZLr4/2QtXHqUKb3Xr/wHZYzwdEb3W5UHrttqM
kI6+70AQwzyKYT7FoVCbjgVIGF2B1tNdpKSjA9ANd4BxctNLHuHb82v82kD4vtYjYqwi7LW1GItF44tqW3D6zt8daBvnHm06L1DiiqPInHzd9305hwPOAIeq
xRIcafudXYfXW4XzJwB3cBct7O5+6czsd0l0RZSvF1svUF8k3m0Dv7XqzyUgdmVah8oLWe9IrePto23nSQoGAQmv+ttv1IaE6s6BGD/grHs43dXnXzy7jp/X
eX+JOiwOBUn8Eng0jnlJ+sK6m7qzgg/K+PlVnF6Yoxboxs8w+P7Nlxd/7ixkPn7WC5DkYA9a2di5J+Wdh9tRo4jpxU6UItvByw75R+043qdVt6fjUbqtXtcj
XsHS3XbR7UkSxXPAOnGMN/50EK99MDuTO65tXEi5kXUpwNNdm8bxP6b4I1vbCCqBTgg58pvnkBiXEHy4TmfWBCfaFANP96MvpFhA1P6b6aQXboyLV+H7abxB
2cSr+C21rQAnvxJyWCIlHc5pKOmv5aY/INMHSgx3EhUKFUXzHOa+fONDcyvuayMh7t26o26iFO6rwlqaOGb36GpzkCIdE+Illm+P0rAoDsbdRnZ1xHGAedX+
P60sFMj512LHZgIhjdf9+s1ijPCgsOEregyQ/efazud4sdEIMYZLU6A0fRDpT2rCgQ0Bp08YffZXFi7pDjf+5L4+UPxgTP03dajOBC2neJJMq3IVzjD7BU0/
fiYIfYLTZ3LOQ5vnnGqO87U2CyU4Ub2X/wFpww5YZRgAAA==
""")))  # embedded from serving/runtime_patches/: 20-moe-routing.py
if SGLANG_RUNTIME_PATCHES:
    site_packages = sorted(Path(VENV).glob("lib/python*/site-packages"))
    assert len(site_packages) == 1, f"Expected one site-packages under {VENV}"
    patch_dir = Path(PREFIX) / "runtime-patches"
    patch_dir.mkdir(parents=True, exist_ok=True)
    for name, text in sorted(SGLANG_RUNTIME_PATCHES.items()):
        digest = hashlib.sha256(text.encode()).hexdigest()[:16]
        marker = patch_dir / f"{name}.{digest}.applied"
        if marker.exists():
            print(f"runtime patch {name} already applied")
            continue
        path = patch_dir / name
        path.write_text(text)
        if name.endswith(".patch"):
            run(["patch", "-p1", "--forward", "--batch", "-d", str(site_packages[0]), "-i", str(path)], env=install_env)
        elif name.endswith(".py"):
            run([PYTHON, str(path), str(site_packages[0])], env=install_env)
        else:
            raise RuntimeError(f"Unknown runtime patch type: {name}")
        marker.write_text("applied\n")
        print(f"runtime patch {name} applied")

# ---- bundled CUDA toolkit and linker symlinks ----
cu13 = sorted(Path(VENV).glob("lib/python*/site-packages/nvidia/cu13"))
if not cu13 or not (cu13[0] / "bin/nvcc").is_file():
    raise RuntimeError("Bundled CUDA 13 nvcc missing; check that the complete Pennyroyal wheelhouse was uploaded")
CUDA_HOME = str(cu13[0])
libdir = Path(CUDA_HOME) / "lib"
lib64 = Path(CUDA_HOME) / "lib64"
if not lib64.exists() and not lib64.is_symlink(): lib64.symlink_to("lib")
for so in sorted(libdir.glob("*.so.*")):
    link = libdir / re.sub(r"\.so\..*$", ".so", so.name)
    if not link.exists() and not link.is_symlink(): link.symlink_to(so.name)
assert (libdir / "libcudart.so").exists(), "libcudart.so missing: JIT linker cannot find CUDA runtime"
if not (libdir / "libcuda.so").exists():
    candidates = []
    for directory in ("/usr/local/nvidia/lib64", "/usr/local/nvidia/lib", "/usr/lib/x86_64-linux-gnu", "/usr/lib64"):
        candidates += sorted(p for p in Path(directory).glob("libcuda.so*") if p.is_file())
    if not candidates:
        code = 'import torch; torch.cuda.init(); print(open("/proc/self/maps").read())'
        maps = run([PYTHON, "-c", code], env=install_env).stdout
        candidates = [Path(p) for p in re.findall(r"(/[^\s]*libcuda\.so[^\s]*)", maps)]
    if not candidates: raise RuntimeError("GPU driver libcuda.so not found")
    link = libdir / "libcuda.so"
    if link.is_symlink(): link.unlink()
    link.symlink_to(candidates[0].resolve())
CXX = next((c for c in ("g++-15", "g++-14", "g++-13", "g++-12", "g++-11", "g++") if shutil.which(c)), None)
assert CXX, "Host C++ compiler not found"
CC = CXX.replace("g++", "gcc")
assert shutil.which(CC), f"Matching C compiler missing: {CC}"

# ---- runtime environment ----
env = dict(os.environ)
env.pop("PYTHONPATH", None)
cache = f"{PREFIX}/cache"
for name in ("huggingface", "torch", "torchinductor", "triton", "flashinfer", "cuda", "sglang/jit"):
    Path(cache, name).mkdir(parents=True, exist_ok=True)
env.update({
    "PYTHONNOUSERSITE": "1", "HF_HUB_OFFLINE": "1", "TRANSFORMERS_OFFLINE": "1",
    "HF_DATASETS_OFFLINE": "1", "UV_OFFLINE": "1", "UV_PYTHON_DOWNLOADS": "never",
    "PYTORCH_CUDA_ALLOC_CONF": "expandable_segments:True",
    "CUDA_HOME": CUDA_HOME, "CUDACXX": f"{CUDA_HOME}/bin/nvcc",
    "PATH": f"{CUDA_HOME}/bin:{VENV}/bin:" + env.get("PATH", ""),
    "LD_LIBRARY_PATH": f"{CUDA_HOME}/lib64:{CUDA_HOME}/lib:" + env.get("LD_LIBRARY_PATH", ""),
    "CC": CC, "CXX": CXX, "CUDAHOSTCXX": CXX,
    "TORCH_CUDA_ARCH_LIST": "12.0",
    "HF_HOME": f"{cache}/huggingface", "XDG_CACHE_HOME": cache,
    "TORCH_HOME": f"{cache}/torch", "CUDA_CACHE_PATH": f"{cache}/cuda",
    "TORCHINDUCTOR_CACHE_DIR": f"{cache}/torchinductor", "TRITON_CACHE_DIR": f"{cache}/triton",
    "FLASHINFER_WORKSPACE_BASE": f"{cache}/flashinfer",
    "SGLANG_CACHE_DIR": f"{cache}/sglang", "SGLANG_JIT_CACHE_DIR": f"{cache}/sglang/jit",
    "SGLANG_ALLOW_OVERWRITE_LONGER_CONTEXT_LEN": "1",
    "OMP_NUM_THREADS": "8", "MKL_NUM_THREADS": "8", "TOKENIZERS_PARALLELISM": "false",
    "MAX_JOBS": "8", "CMAKE_BUILD_PARALLEL_LEVEL": "8",
    "FLASHINFER_NINJA_JOBS": "8", "FLASHINFER_NVCC_THREADS": "2", "TORCHINDUCTOR_COMPILE_THREADS": "8",
    "SGLANG_ENABLE_SM120_LOWM_BF16_GEMM": "1", "SGLANG_SM120_ONLINE_MXFP8": "0",
    "SGLANG_SM120_LOWM_FP8_WEIGHT": "0", "SGLANG_SM120_LM_HEAD_FP8": "0",
    "SGLANG_MAMBA_CONV_DTYPE": "bfloat16", "SGLANG_NUMA_BIND_V2": "false",
    "SGLANG_MM_PREPROCESS_DEVICE": "cpu", "NUMPY_MADVISE_HUGEPAGE": "0",
})
run([str(Path(CUDA_HOME) / "bin/nvcc"), "--version"], env=env)
print("Host compiler:", CXX)
# Report the optional later caching patch. This cell does not modify wheel code.
run([PYTHON, "-c", '''import pathlib, sglang, torch
print("sglang", sglang.__version__, "| torch", torch.__version__)
assert torch.cuda.is_available(), "CUDA unavailable"
print("GPU:", torch.cuda.get_device_name(0), "capability:", torch.cuda.get_device_capability(0))
'''], env=env)

# ---- JIT caches (issue #6): describe this runtime, restore a matching snapshot ----
# Any failure here leaves the caches empty (a cold start, as without the flags).
JIT_CACHE_MANIFEST = None
if CFG["JIT_CACHE_SAVE"] or CFG["JIT_CACHE_DIR"]:
    JIT_CACHE_ROOTS = jit_cache.cache_roots(cache, env.get("HOME") or Path.home())
    try:
        jit_probe = Path(PREFIX) / "jit_cache_probe.py"
        jit_probe.write_text(jit_cache.RUNTIME_PROBE)
        JIT_CACHE_MANIFEST = jit_cache.make_manifest(
            jit_cache.parse_probe(run([PYTHON, jit_probe], env=env).stdout),
            jit_cache.config_snapshot(CFG, env, Path(MODEL_DIR, "config.json").read_bytes()),
            JIT_CACHE_ROOTS, label=os.environ.get("TAAF_ARM"))
        print("JIT cache config hash:", JIT_CACHE_MANIFEST["config_hash"])
    except Exception as exc:
        print(f"WARNING: JIT cache disabled, the runtime probe failed ({type(exc).__name__}: {exc})")
if CFG["JIT_CACHE_DIR"] and JIT_CACHE_MANIFEST is not None:
    jit_t0 = time.time()
    try:
        jit_restored, jit_message = jit_cache.restore(
            CFG["JIT_CACHE_DIR"], JIT_CACHE_ROOTS, JIT_CACHE_MANIFEST, Path(PREFIX) / "jit-cache-staging")
    except Exception as exc:
        jit_restored, jit_message = False, f"{type(exc).__name__}: {exc}"
    print(f"JIT cache {'restored' if jit_restored else 'NOT restored, starting cold'} "
          f"in {time.time()-jit_t0:.1f}s: {jit_message}")

# ---- validate target and create the read-only draft's SGLang view ----
target_cfg = json.loads(Path(MODEL_DIR, "config.json").read_text())
q = target_cfg.get("quantization_config", {})
assert q.get("quant_method") == "auto-round" and q.get("bits") == 4, "MODEL_DIR must be the Intel/albucino AutoRound INT4 checkpoint"
_, target_shards = indexed_shards(MODEL_DIR)
print("Target:", MODEL_DIR, "| shards:", len(target_shards), "| PLE dtype:", target_cfg.get("text_config", {}).get("ple_embedding_dtype", "checkpoint default (typically BF16)"))
print("MoE experts per token (checkpoint):", target_cfg.get("text_config", {}).get("num_experts_per_tok"),
      "| routing variant:", env.get("SGLANG_ARC_MOE_ROUTING") or "default")
DRAFT_VIEW = None
if CFG["SPEC"]:
    assert Path(DRAFT_MODEL_DIR).is_dir(), f"draft dir not found: {DRAFT_MODEL_DIR}"
    DRAFT_VIEW = prepare_draft_view(MODEL_DIR, DRAFT_MODEL_DIR, PREFIX)

# ---- assemble Pennyroyal / AutoRound launch arguments ----
graph_bs = sorted({1, 2, 4, 7, 8, 9, 10, CFG["MAXREQ"], CFG["CUDAGRAPH_MAXBS"], *CFG["CUDAGRAPH_BS_EXTRA"]})
graph_bs = [n for n in graph_bs if n <= CFG["CUDAGRAPH_MAXBS"]]
args = [SGLANG, "serve", "--model-path", MODEL_DIR, "--load-format", "safetensors",
    "--model-loader-extra-config", '{"enable_multithread_load":false}',
    "--served-model-name", str(CFG["SERVED_NAME"]), "--host", "127.0.0.1",
    "--port", str(SERVED_MODEL_PORT), "--tensor-parallel-size", "1",
    "--dtype", "bfloat16", "--quantization", "auto-round",
    "--kv-cache-dtype", CFG["KVDTYPE"], "--mem-fraction-static", str(CFG["MEMFRAC"]),
    "--context-length", str(CFG["CTX"]), "--page-size", "64",
    "--max-running-requests", str(CFG["MAXREQ"]), "--chunked-prefill-size", str(CFG["CHUNK"]),
    "--max-prefill-tokens", str(CFG["MAX_PREFILL"]),
    "--cuda-graph-max-bs-decode", str(CFG["CUDAGRAPH_MAXBS"]),
    "--cuda-graph-bs-decode", *map(str, graph_bs),
    "--mamba-ssm-dtype", CFG["SSM_DTYPE"], "--max-mamba-cache-size", str(CFG["MAMBA_CACHE"]),
    "--mamba-radix-cache-strategy", CFG["MAMBA_RADIX"], "--mamba-track-interval", "64",
    "--mamba-backend", CFG["LINEAR_BACKEND"],
    "--linear-attn-decode-backend", CFG["LINEAR_BACKEND"],
    "--linear-attn-prefill-backend", CFG["LINEAR_BACKEND"],
    "--moe-runner-backend", "auto", "--ple-offload-embedding", "--trust-remote-code",
    "--mm-feature-transport", "cpu", "--image-processor-backend", "pil",
    "--reasoning-parser", "qwen3", "--tool-call-parser", "qwen3_coder",
    "--default-chat-template-kwargs", '{"preserve_thinking":true}',
    "--watchdog-timeout", "1800", "--schedule-policy", "lpm", "--warmups", "structured_output",
    "--enable-cache-report", "--enable-metrics", "--enable-request-time-stats-logging"]
if CFG["ATTN_BACKEND"]: args += ["--attention-backend", CFG["ATTN_BACKEND"]]
if CFG["TEXT_CONFIG_OVERRIDE"]:
    _view = Path(PREFIX) / "target-view"
    _view.mkdir(parents=True, exist_ok=True)
    for _src in Path(MODEL_DIR).iterdir():
        _dst = _view / _src.name
        if _src.name != "config.json" and not _dst.is_symlink():
            _dst.symlink_to(_src.resolve())
    _cfg = json.loads(Path(MODEL_DIR, "config.json").read_text())
    _cfg["text_config"].update(CFG["TEXT_CONFIG_OVERRIDE"])
    (_view / "config.json").write_text(json.dumps(_cfg, indent=2))
    args[args.index("--model-path") + 1] = str(_view)
    print("Target view", _view, "text_config override:", CFG["TEXT_CONFIG_OVERRIDE"])
if CFG["SPEC_ADAPTIVE"]:
    _acfg = Path(PREFIX) / "adaptive_spec.json"
    _acfg.write_text(json.dumps(CFG["SPEC_ADAPTIVE"]))
    args += ["--speculative-adaptive", "--speculative-adaptive-config", str(_acfg)]
    print("Adaptive speculation:", CFG["SPEC_ADAPTIVE"])
if CFG["PREFETCH_CHECKPOINTS"]: args += ["--weight-loader-prefetch-checkpoints"]
if CFG["DROP_CACHE_AFTER_LOAD"]: args += ["--weight-loader-drop-cache-after-load"]
tmpl = Path(MODEL_DIR, "chat_template.jinja")
if tmpl.is_file(): args += ["--chat-template", str(tmpl)]
if not CFG["AUTOTUNE"]: args += ["--disable-flashinfer-autotune"]
if CFG["GDN_MTP_CACHE_MODE"]: args += ["--gdn-mtp-cache-mode", CFG["GDN_MTP_CACHE_MODE"]]
if CFG["SPEC"]:
    args += ["--speculative-algorithm", "NEXTN", "--speculative-num-steps", str(CFG["SPEC_STEPS"]),
        "--speculative-eagle-topk", "1", "--speculative-num-draft-tokens", str(CFG["SPEC_STEPS"]+1),
        "--speculative-draft-model-path", str(DRAFT_VIEW),
        "--speculative-draft-model-quantization", "compressed-tensors", "--speculative-moe-runner-backend", "auto",
        "--speculative-draft-kv-cache-dtype", CFG["DRAFT_KVDTYPE"],
        "--speculative-accept-threshold-single", str(CFG["SPEC_ACCEPT_SINGLE"]),
        "--speculative-accept-threshold-acc", str(CFG["SPEC_ACCEPT_ACC"])]
    if CFG["FRSPEC"]:
        tok = find_unique(WHEELHOUSE_DIR, "hot_tokens_64k.pt", required=False)
        if tok is None: tok = find_unique(WHEELHOUSE_DIR, "flash-next-64k.pt")
        assert sha256(tok) == TOKEN_MAP_SHA, "Wrong FR-Spec map; use the map from the pinned Pennyroyal bundle"
        assert sha256(Path(MODEL_DIR, "tokenizer.json")) == TOKENIZER_SHA, "Tokenizer differs from the FR-Spec map; set CFG['FRSPEC']=False"
        args += ["--speculative-token-map", str(tok)]

# ---- launch detached and wait for health ----
precache_model_thread.start()
print("\n" + shlex.join(args) + "\n")
log_offset = Path(LOG).stat().st_size if Path(LOG).exists() else 0
with open(LOG, "ab", buffering=0) as logf:
    proc = subprocess.Popen(args, env=env, stdout=logf, stderr=subprocess.STDOUT, start_new_session=True)
print(f"pid {proc.pid} -> {LOG}")

def show_log_tail():
    with open(LOG, "rb") as f:
        f.seek(0, 2)
        f.seek(max(log_offset, f.tell()-20000))
        print(f.read().decode(errors="replace"))

import threading


def _serve_log_last_line():
    try:
        with open(LOG, "rb") as f:
            f.seek(0, 2)
            f.seek(max(log_offset, f.tell() - 4000))
            lines = [l for l in f.read().decode(errors="replace").splitlines() if l.strip()]
        return lines[-1][-300:] if lines else "(empty)"
    except OSError as exc:
        return f"(unreadable: {exc})"


def _stream_serve_log_until_ready():
    # Runs after the deadline released the benchmark: report the server's own
    # progress every 30 s until /health answers or the process exits.
    global SERVER_READY_TIME
    seen = Path(LOG).stat().st_size if Path(LOG).exists() else 0
    while proc.poll() is None:
        try:
            with urllib.request.urlopen(url, timeout=5) as r:
                if r.status == 200:
                    SERVER_READY_TIME = time.time()
                    print(f"[serve] READY after {int(time.time()-t0)}s", flush=True)
                    return
        except Exception:
            pass
        with open(LOG, "rb") as f:
            f.seek(seen)
            fresh = f.read()
            seen = f.tell()
        for line in fresh.decode(errors="replace").splitlines()[-15:]:
            if line.strip():
                print(f"[serve] {line[-300:]}", flush=True)
        time.sleep(30)
    print(f"[serve] server process exited with {proc.returncode}", flush=True)
    show_log_tail()


url, t0 = f"http://127.0.0.1:{SERVED_MODEL_PORT}/health", time.time()
last_report = -30
while time.time() - NOTEBOOK_START_TIME < SERVER_STARTUP_TIMEOUT:
    if proc.poll() is not None:
        show_log_tail()
        raise RuntimeError(f'SGLang exited with code {proc.returncode} during startup; see serve.log')
    try:
        with urllib.request.urlopen(url, timeout=5) as r:
            if r.status == 200:
                SERVER_READY_TIME = time.time(); print(f"\nREADY after {int(time.time()-t0)}s -> {url.replace('/health','/v1')}")
                break
    except Exception:
        pass
    elapsed = int(time.time()-t0)
    if elapsed-last_report >= 30:
        print(f"  {elapsed}s ... serve.log: {_serve_log_last_line()}", flush=True); last_report=elapsed
    time.sleep(3)
else:
    print(f"\nDEADLINE at {int(time.time()-NOTEBOOK_START_TIME)}s from notebook start; "
          f"releasing the benchmark with the server still loading")
    show_log_tail()
    threading.Thread(target=_stream_serve_log_until_ready, daemon=True).start()

# ---- JIT caches (issue #6): snapshot them once the server is ready ----
# Ready means warm: SGLang answers /health only after CUDA-graph capture and its
# warmup request. A server still loading at the deadline is saved once ready.
if CFG["JIT_CACHE_SAVE"] and JIT_CACHE_MANIFEST is not None:
    import threading

    def _jit_server_ready():
        try:
            with urllib.request.urlopen(url, timeout=5) as r:
                return r.status == 200
        except Exception:
            return False

    def _save_jit_cache():
        tar_path = Path(WORKING_DIR) / jit_cache.TAR_NAME
        started = time.time()
        try:
            saved = jit_cache.save(tar_path, JIT_CACHE_ROOTS, JIT_CACHE_MANIFEST)
        except Exception as exc:
            print(f"WARNING: JIT cache NOT saved ({type(exc).__name__}: {exc})", flush=True)
            return
        print(f"JIT cache saved to {tar_path} in {time.time()-started:.1f}s: {saved['files']} files, "
              f"{saved['tar_bytes']/1e6:.0f} MB, files per root {saved['files_per_root']}", flush=True)
        for directory, size in jit_cache.unsaved_dirs(cache, JIT_CACHE_ROOTS):
            print(f"  not in the snapshot: {directory} ({size/1e6:.1f} MB)", flush=True)

    def _save_jit_cache_when_ready():
        while proc.poll() is None:
            if _jit_server_ready():
                _save_jit_cache()
                return
            time.sleep(10)
        print("WARNING: JIT cache NOT saved: the server exited before it was ready", flush=True)

    if _jit_server_ready():
        _save_jit_cache()
    else:
        threading.Thread(target=_save_jit_cache_when_ready, daemon=True).start()


## 6. Load the benchmark

Unpickle the deployment target and the benchmark, stamping the real submission state onto the
target and pointing the benchmark's outputs at the Kaggle working directory.

In [ ]:
# Restore the deployment target and record the real submission state on it.
with open(BUNDLE_DIR / "deploy_target.pkl", "rb") as file:
    target = pickle.load(file)
target.actual_run_as_submission = TRUE_SUBMISSION
target.is_competition_rerun = TRUE_SUBMISSION

# Restore the benchmark and point its outputs at the Kaggle working dir.
with open(BUNDLE_DIR / "benchmark_initial.pkl", "rb") as file:
    bm = pickle.load(file)
bm.job_dir = WORKING_DIR

## 7. Customization hook

Optional: tweak `bm`, `bm.games`, or `bm.solver` here before the run starts — the safe place
for one-off experiments once the deployed bundle has loaded.

In [ ]:
# Make one-off changes to `bm`, `bm.games`, or `bm.solver` here before the run starts.
bm.solver.max_runtime_s_per_game = 48*60 #7920.0
bm.solver.analyzer_timeout = 900.0
bm.solver.concurrency = 10
bm.solver.max_actions_per_game = None
bm.solver.save_request_logs = True
bm.n_passes = 1
bm.game_weights = None

demo_excluded_games = []

if TRUE_SUBMISSION:  # overrides for contest run
    bm.solver.save_request_logs = False
    bm.n_passes = 1
    if USE_PRIORITY_SCHEDULING:
        bm.solver.concurrency = 120  # game count should be 110
        bm.solver.max_runtime_s_per_game = 532*60
else:
    if USE_PRIORITY_SCHEDULING:
        bm.solver.concurrency = (25 - len(demo_excluded_games)) * bm.n_passes
        bm.solver.max_runtime_s_per_game = int(121 * 60)
    if float(getattr(target, 'max_runtime_s', 0.0) or 0.0) != 32400.0:
        raise RuntimeError(
            f'Expected the 32400-second notebook budget, got {target.max_runtime_s!r}.'
        )

# show config
print('benchmark.solver:', bm.solver)

## 8. Run the system monitor

Periodically show free main memory, VRAM and disk space.

In [ ]:
# --- system census, one line every 10 minutes -------------------------------
# Everything reads as used/total. RAM is total - MemAvailable, which runs higher
# than the "used" column in `free -h`: that one excludes reclaimable page cache,
# and MemAvailable is the figure that actually predicts an OOM. Disk likewise
# counts the root reserve as unavailable, matching df.
#
# A daemon thread rather than a hook in the run loop: the point of watching a
# long run is to see what happens when it stops behaving, and a print driven by
# the harness stops with it. This keeps reporting through a deadlock.
import os, subprocess, threading, sys

def _sysline(paths=("/", "/kaggle/working")):
    GB = 1024 ** 3
    out = []
    mem = {}
    try:
        for line in open("/proc/meminfo"):
            k, _, rest = line.partition(":")
            p = rest.split()
            if p:
                mem[k] = int(p[0]) * 1024
    except OSError:
        pass
    if mem.get("MemTotal") and mem.get("MemAvailable") is not None:
        out.append(f"RAM {(mem['MemTotal']-mem['MemAvailable'])/GB:.1f}/{mem['MemTotal']/GB:.1f}")
    if mem.get("SwapTotal"):
        out.append(f"swap {(mem['SwapTotal']-mem['SwapFree'])/GB:.1f}/{mem['SwapTotal']/GB:.1f}")
    else:
        out.append("swap none")
    st = {}
    try:
        for line in open("/proc/self/status"):
            k, _, rest = line.partition(":")
            p = rest.split()
            if k in ("VmRSS", "Threads") and p:
                st[k] = int(p[0])
    except OSError:
        pass
    proc = []
    if "VmRSS" in st:
        proc.append(f"{st['VmRSS']*1024/GB:.1f} GB")
    proc.append(f"{st.get('Threads', threading.active_count())} threads")
    try:
        proc.append(f"{len(os.listdir('/proc/self/fd'))} fds")
    except OSError:
        pass
    out.append("proc " + ", ".join(proc))
    try:
        r = subprocess.run(["nvidia-smi", "--query-gpu=memory.used,memory.total",
                            "--format=csv,noheader,nounits"],
                           capture_output=True, text=True, timeout=5, check=False)
        gpus = []
        for i, line in enumerate(r.stdout.splitlines()):
            p = [x.strip() for x in line.split(",")]
            if len(p) == 2:
                gpus.append(f"GPU{i} {float(p[0])/1024:.1f}/{float(p[1])/1024:.1f}")
        out.append(" ".join(gpus) if gpus else "GPU unavailable")
    except Exception:
        out.append("GPU unavailable")          # nvidia-smi hangs on a wedged GPU
    disks = []
    for path in paths:
        try:
            s = os.statvfs(path)
        except OSError:
            continue
        total = s.f_blocks * s.f_frsize
        disks.append(f"{path} {(total - s.f_bavail*s.f_frsize)/GB:.1f}/{total/GB:.1f}")
    if disks:
        out.append("disk " + ", ".join(disks))
    return "[sys] " + " | ".join(out)


def start_monitor(interval_minutes=10.0, paths=("/", "/kaggle/working")):
    stop = threading.Event()
    def loop():
        while not stop.is_set():
            try:
                print(_sysline(paths), flush=True)
            except Exception as exc:
                print(f"[sys] census failed: {exc!r}", flush=True)
            stop.wait(interval_minutes * 60.0)
    threading.Thread(target=loop, name="system-stats", daemon=True).start()
    return stop

if not TRUE_SUBMISSION:
    _stop_monitor = start_monitor(2)
    print(_sysline())

## 8b. Log the score timeline

Every minute, append each game's levels, actions, tokens and official score to `/kaggle/working/score_timeline.jsonl`, with wall-clock time since the notebook started, since the model server was ready and since the benchmark started; every 5 minutes print a `[score]` line. Read-only: it never touches the solver, the games or the server.

In [ ]:
# --- score timeline: one JSON line per game every minute ----------------------
# kaggle/build_notebook.py pastes this file into its own notebook cell (after the
# system monitor), followed by a call to start_score_logger. It is also imported by
# kaggle/tests/test_score_logger.py, so it must stay standard-library only.
#
# A daemon thread reads the live taaf GameRun objects in bm.game_runs (Benchmark.run
# appends one per game when it starts them, and the framework updates
# history / actions_per_level / levels_completed / state in place as moves are made).
# It only reads: no locks, no calls into the solver, the engine or the model server,
# so it cannot change a game decision, a prompt or a request. Every read is wrapped,
# so a half-updated GameRun or a write error costs at most one line, never the run.
#
# Every `interval_s` it appends to `path` one line per game:
#   {"kind": "game", "t_notebook_s", "t_ready_s", "t_bench_s", "wall", "game", "pass",
#    "levels", "n_levels", "actions", "score", "tokens", "state", "run_state"}
# and one {"kind": "run", ..., "mean_score", "levels", "actions", "tokens",
# "games_finished", "n_games"} line. Every `print_every_s` it prints
#   [score] t=37m nb=49m mean=12.40 lv=31 act=1234 tok=1.23M fin=3/25 | ft09 3L 40.1 | ...
# t is minutes since the benchmark started, nb since the notebook started.
# score is the official formula (taaf GameRun._compute_final_score): level score
# min(115, (baseline/actions)^2 * 100), level i weighted by i, uncleared levels 0,
# capped at 100 * (weight of scored levels) / (total weight). None when the engine
# hides the baselines (competition rerun).
import json
import threading
import time
from datetime import datetime


def _official_score(number_of_levels, base_actions_per_level, levels_completed, actions_per_level):
    if base_actions_per_level is None or number_of_levels == 0:
        return None
    total_score = 0.0
    total_weights = 0
    max_weights = 0
    for level_idx in range(number_of_levels):
        weight = level_idx + 1
        total_weights += weight
        actions = actions_per_level[level_idx] if level_idx < len(actions_per_level) else 0
        if level_idx < levels_completed and actions > 0:
            level_score = min(115.0, (base_actions_per_level[level_idx] / actions) ** 2 * 100)
        else:
            level_score = 0.0
        if level_score > 0:
            max_weights += weight
        total_score += level_score * weight
    if total_weights == 0:
        return 0.0
    return min(total_score / total_weights, max_weights / total_weights * 100)


_FINISHED_STATES = {"won", "gave_up", "cancelled"}


class _TokenCounter:
    """Running sum of generated tokens per GameRun: each tick reads only new history records."""

    def __init__(self):
        self._seen = {}  # id(run) -> (records counted, tokens)

    def tokens(self, run, n_records):
        count, total = self._seen.get(id(run), (0, 0))
        if n_records < count:  # not expected (history only grows); recount
            count, total = 0, 0
        hist = run.history
        for i in range(count, n_records):
            total += int(getattr(hist[i], "generated_tokens", 0) or 0)
        self._seen[id(run)] = (n_records, total)
        return total


def score_snapshot(bm, *, notebook_start=None, server_ready=None, now=None, counter=None):
    """One dict per game plus a final "run" dict, read from the live benchmark. Never raises
    for a single game: a game that cannot be read gets {"kind": "game", "error": ...}."""
    now = time.time() if now is None else now
    runs = list(getattr(bm, "game_runs", None) or [])
    start = getattr(bm, "start_time", None)
    clock = {
        "wall": datetime.fromtimestamp(now).isoformat(timespec="seconds"),
        "t_notebook_s": None if notebook_start is None else round(now - notebook_start, 1),
        "t_ready_s": None if server_ready is None else round(now - server_ready, 1),
        "t_bench_s": None if start is None else round((datetime.fromtimestamp(now) - start).total_seconds(), 1),
    }
    counter = counter or _TokenCounter()
    n_passes = max(1, int(getattr(bm, "n_passes", 1) or 1))
    per_pass = len(runs) // n_passes if runs and len(runs) % n_passes == 0 else len(runs)
    rows = []
    for i, run in enumerate(runs):
        row = {"kind": "game", **clock, "game": str(getattr(run, "game_id", f"#{i}")),
               "pass": i // per_pass if per_pass else 0}
        try:
            # levels before the counts: a level cleared between the two reads then
            # scores on the next tick instead of with a short action count.
            levels = int(run.levels_completed)
            per_level = list(run.actions_per_level)
            n_records = len(run.history)
            raw_state = str(run.state)
            final = run.final_score
            score = float(final) if final is not None else _official_score(
                int(run.number_of_levels), run.base_actions_per_level, levels, per_level)
            row.update({
                "levels": levels, "n_levels": int(run.number_of_levels), "actions": n_records,
                "score": None if score is None else round(score, 4),
                "tokens": counter.tokens(run, n_records),
                "state": ("playing" if raw_state == "playing" else
                          "finished" if raw_state in _FINISHED_STATES else raw_state),
                "run_state": raw_state,
            })
        except Exception as exc:  # a half-built or odd GameRun: report, keep going
            row["error"] = repr(exc)[:200]
        rows.append(row)
    ok = [r for r in rows if "error" not in r]
    by_game = {}
    for r in ok:
        by_game.setdefault(r["game"], []).append(r["score"] or 0.0)
    mean = (sum(sum(v) / len(v) for v in by_game.values()) / len(by_game)) if by_game else None
    rows.append({
        "kind": "run", **clock,
        "mean_score": None if mean is None else round(mean, 4),
        "levels": sum(r["levels"] for r in ok), "actions": sum(r["actions"] for r in ok),
        "tokens": sum(r["tokens"] for r in ok),
        "games_finished": sum(1 for r in ok if r["state"] != "playing"),
        "n_games": len({r["game"] for r in rows if r["kind"] == "game"}),
        "unreadable": len(rows) - len(ok),
    })
    return rows


def format_score_line(rows):
    """[score] t=37m nb=49m mean=12.40 lv=31 act=1234 tok=1.23M fin=3/25 | ft09 3L 40.1 | ..."""
    run = rows[-1]

    def minutes(s):
        return "?" if s is None else f"{s / 60:.0f}m"

    def num(v):
        return "?" if v is None else f"{v:.1f}"

    head = (f"[score] t={minutes(run['t_bench_s'])} nb={minutes(run['t_notebook_s'])} "
            f"mean={'?' if run['mean_score'] is None else format(run['mean_score'], '.2f')} "
            f"lv={run['levels']} act={run['actions']} tok={run['tokens'] / 1e6:.2f}M "
            f"fin={run['games_finished']}/{run['n_games']}")
    games = []
    for r in sorted((r for r in rows if r["kind"] == "game"), key=lambda r: (r["game"], r["pass"])):
        name = r["game"].split("-")[0] + (f"#{r['pass']}" if r["pass"] else "")
        if "error" in r:
            games.append(f"{name} ?")
        else:
            mark = "" if r["state"] == "playing" else ("*" if r["state"] == "finished" else "!")
            games.append(f"{name} {r['levels']}L {num(r['score'])}{mark}")
    return " | ".join([head, *games])


def start_score_logger(get_bm, path, *, notebook_start=None, get_server_ready=None,
                       interval_s=60.0, print_every_s=300.0, out=print):
    """Start the daemon thread; returns an Event that stops it. `get_bm` and
    `get_server_ready` are called every tick, so they may return None until bm exists /
    the server answers. Logging starts when bm.game_runs is non-empty and ends one tick
    after bm.end_time is set (once a run has been seen in progress). The thread is
    the returned Event's `.thread`."""
    stop = threading.Event()
    counter = _TokenCounter()
    state = {"last_print": None, "errors": 0, "running": False}

    def tick():
        bm = get_bm()
        if bm is None or getattr(bm, "start_time", None) is None or not getattr(bm, "game_runs", None):
            return False
        # Benchmark.run clears end_time when it starts; only stop after seeing a run in progress.
        if getattr(bm, "end_time", None) is None:
            state["running"] = True
        elif not state["running"]:
            return False
        ready = get_server_ready() if get_server_ready else None
        rows = score_snapshot(bm, notebook_start=notebook_start, server_ready=ready, counter=counter)
        with open(path, "a", encoding="utf-8") as f:
            f.write("".join(json.dumps(r, separators=(",", ":")) + "\n" for r in rows))
        now = time.monotonic()
        ended = getattr(bm, "end_time", None) is not None
        if ended or state["last_print"] is None or now - state["last_print"] >= print_every_s - 1:
            out(format_score_line(rows), flush=True)
            state["last_print"] = now
        return ended

    def loop():
        while not stop.is_set():
            try:
                if tick():
                    return
            except Exception as exc:
                state["errors"] += 1
                if state["errors"] <= 3:  # say it, but never flood the log
                    out(f"[score] logger tick failed ({state['errors']}): {exc!r}", flush=True)
            stop.wait(interval_s)

    stop.thread = threading.Thread(target=loop, name="score-timeline", daemon=True)
    stop.thread.start()
    return stop

if not TRUE_SUBMISSION:
    _stop_score_logger = start_score_logger(
        lambda: globals().get("bm"),
        WORKING_DIR / "score_timeline.jsonl",
        notebook_start=NOTEBOOK_START_TIME,
        get_server_ready=lambda: globals().get("SERVER_READY_TIME"),
    )
    print("score timeline -> /kaggle/working/score_timeline.jsonl (every 60 s, [score] line every 5 min)")


## 9. Run the benchmark

In a real competition rerun (`KAGGLE_IS_COMPETITION_RERUN`), wait for the Kaggle gateway and
play the **live competition Arcade**. Otherwise — an interactive "Save & Run" — play the
competition's **bundled environment files offline**, with no gateway required, so the notebook
runs end-to-end without a submission. Teardown commands run afterward even if the run raises.

In [ ]:
print('Starting benchmark...')

# Build the live competition game list from the gateway's available environments.
def _competition_games():
    import arc_agi

    import taaf.game_api

    spec = taaf.game_api.ArcadeSpec(
        operation_mode=arc_agi.OperationMode.COMPETITION,
        arc_base_url=os.environ["ARC_BASE_URL"],
        environments_dir="",
    )
    arcade = arc_agi.Arcade(
        operation_mode=arc_agi.OperationMode.COMPETITION,
        arc_base_url=spec.arc_base_url,
        environments_dir="",
    )
    game_ids = [env_info.game_id for env_info in arcade.available_environments]
    if not game_ids:
        raise RuntimeError("Competition Arcade exposed zero environments.")
    return [taaf.game_api.GameAPI(env_name=game_id, arcade_spec=spec) for game_id in game_ids]


# Build the offline game list from the competition's bundled environment files.
def _offline_games(env_dir: str):
    import arc_agi

    import taaf.game_api

    spec = taaf.game_api.ArcadeSpec(operation_mode=arc_agi.OperationMode.OFFLINE, environments_dir=env_dir)
    arcade = arc_agi.Arcade(operation_mode=arc_agi.OperationMode.OFFLINE, environments_dir=env_dir)
    game_ids = [env_info.game_id for env_info in arcade.available_environments]
    if not game_ids:
        raise RuntimeError(f"No offline environments found under {env_dir}.")
    game_ids = [game_id for game_id in game_ids if not any(game_id.startswith(g) for g in demo_excluded_games)]
    return [taaf.game_api.GameAPI(env_name=game_id, arcade_spec=spec) for game_id in game_ids]


# The gateway can take a while to come up; poll until it answers.
def _wait_for_gateway(base_url: str, timeout_s: float = 600.0) -> None:
    deadline = time.monotonic() + timeout_s
    last_error = ""
    while time.monotonic() < deadline:
        try:
            with urlopen(f"{base_url}api/games", timeout=10) as response:
                if response.status < 500:
                    return
        except Exception as exc:
            last_error = repr(exc)
        time.sleep(5)
    raise RuntimeError(f"Kaggle gateway did not become ready: {last_error}")


# Print the run preamble and persist the launcher's git status for diagnostics.
#print((BUNDLE_DIR / "preamble.txt").read_text())
(WORKING_DIR / "git_status.txt").write_text((BUNDLE_DIR / "git_status.txt").read_text())

# arc_agi reads RECORDINGS_DIR and ARC_API_KEY from env (ArcadeSpec carries neither); operation
# mode, environments dir, and base url are all passed explicitly via the spec, so no env is needed.
os.environ.setdefault("RECORDINGS_DIR", str(WORKING_DIR / "server_recording"))

if TRUE_SUBMISSION:
    # Real submission: play the live competition Arcade served by the Kaggle gateway.
    os.environ.setdefault("ARC_API_KEY", "test-key-123")
    os.environ.setdefault("ARC_BASE_URL", "http://gateway:8001/")
    # The gateway boots asynchronously; wait before swapping in its game list.
    _wait_for_gateway(os.environ["ARC_BASE_URL"])
    bm.games = _competition_games()
else:
    # Interactive run: play the bundled competition environments offline (no gateway).
    # The competition's environment files ship alongside the wheelhouse in the competition dataset.
    competition_env_files = str(Path(_resolve_input("/kaggle/input/competitions/arc-prize-2026-arc-agi-3/arc_agi_3_wheels")).parent / "environment_files")
    bm.games = _offline_games(competition_env_files)

# Outside a real submission, stop ~10 min before the wall-clock budget for a graceful exit.
soft_end = None
if not TRUE_SUBMISSION:
    budget = float(getattr(target, "max_runtime_s", 0.0) or 0.0)
    if budget > 0:
        soft_end = datetime.fromtimestamp(NOTEBOOK_START_EPOCH) + timedelta(seconds=budget - min(600.0, budget / 2))

# Play the benchmark; teardown commands run even if the run raises.
await bm.run(soft_end_time=soft_end, runtime_environment=target, minimal_diagnostics=TRUE_SUBMISSION)
if not TRUE_SUBMISSION:
    # An offline run isn't scored, but Kaggle still expects a submission.parquet output.
    import pandas as pd

    pd.DataFrame(
        [["1_0", "1", True, 1]],
        columns=["row_id", "game_id", "end_of_game", "score"],
    ).to_parquet(WORKING_DIR / "submission.parquet", index=False)

## 10. Show the diagnostics

A non-submission run writes `diagnostics.html` to `/kaggle/working`; it is rendered inline below
(and downloadable from the working directory). You should be able to click around through the links.

In [ ]:
from html import escape

from IPython.display import HTML, display

diagnostics_html = WORKING_DIR / "diagnostics.html"
if diagnostics_html.is_file():
    # Isolate the full document in an iframe so its styles don't leak into the notebook.
    display(
        HTML(
            f'<iframe srcdoc="{escape(diagnostics_html.read_text(), quote=True)}" '
            'width="100%" height="900" style="border:0"></iframe>'
        )
    )
else:
    print("No diagnostics.html — minimal diagnostics (real submission) suppresses it.")